# Do the missingness families hold on this data?

> **Provenance.** The families are transcribed from Deotte, 'EDA for Columns V and ID'. See ATTRIBUTION.md.

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [ ]:

import plotly.express as px
import polars as pl
from IPython.display import Markdown, display

pl.Config.set_tbl_rows(100)

def get_raw_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
v_cols = [c for c in df.columns if c.startswith("V")]


### Missingness Families

Grouping the 339 columns by exact null count yields 15 distinct families.

In [ ]:
null_counts = [(c, df[c].null_count()) for c in v_cols]
df_nulls = pl.DataFrame(null_counts, schema=["column", "null_count"], orient="row")

family_sizes = (
    df_nulls.group_by("null_count")
    .agg(pl.len().alias("family_size"))
    .sort("family_size", descending=True)
)

fig = px.bar(family_sizes.to_pandas(), x="null_count", y="family_size", 
             title="Missingness Families: Size by Null Count",
             labels={"null_count": "Exact Null Count", "family_size": "Number of Columns in Family"},
             template="plotly_white")
fig.update_xaxes(type='category')
fig.show()

families = (
    df_nulls.group_by("null_count")
    .agg(pl.col("column").alias("cols"))
    .sort("null_count", descending=True)
    .to_dict(as_series=False)
)

table = (
    "| Null count | Family size | Columns |\n"
    "| --- | ---: | --- |\n"
    + "\n".join(
        f"| {null_count} | {len(cols)} | {', '.join(cols)} |"
        for null_count, cols in zip(families["null_count"], families["cols"])
    )
)
display(Markdown(table))

missingness_families = (
    df_nulls.group_by("null_count")
    .agg(pl.col("column").alias("cols"))
    .to_dict(as_series=False)
)
missingness_map = {c: cols for cols in missingness_families['cols'] for c in cols}
